# Buổi G6: Học khi ít nhãn, tận dụng dữ liệu chưa gán nhãn

**Hiện trường.** Chị Hạnh có 1.347 ảnh hóa đơn viết tay cần phân loại theo chữ số, nhưng chỉ có thời gian gán nhãn bằng tay **30 ảnh**. 1.317 ảnh còn lại không có nhãn. Có hai câu hỏi: *30 nhãn thì mô hình làm được đến đâu?* và *1.317 ảnh chưa gán nhãn có giúp được gì không?*

Buổi này dạy **học bán giám sát** (*semi-supervised learning*): dùng cả dữ liệu có nhãn và không nhãn.
- **Tự dạy** (*self-training*): mô hình đoán nhãn cho ảnh chưa gán, giữ những đoán chắc chắn làm nhãn mới, học lại.
- **Lan truyền nhãn** (*label spreading*): nhãn "chảy" từ ảnh đã gán sang ảnh hàng xóm giống nó.
- **Học chủ động** (*active learning*): chọn ảnh nào để gán nhãn tiếp theo cho đáng công nhất.

**Ẩn dụ:** lớp có 30 học sinh có sổ điểm, còn 1.300 bạn chưa biết điểm. Nếu một bạn mới ngồi sát nhóm toàn học sinh giỏi, đoán bạn ấy giỏi là hợp lý: **hàng xóm giống nhau thường cùng nhãn**.

**Mindset của người làm dữ liệu:** dữ liệu chưa nhãn không miễn phí: nếu đoán sai rồi học theo cái sai, lỗi sẽ tự khuếch đại. Mọi kết quả phải đo trên **tập thi có nhãn thật**.

**Lưu ý:** dùng `load_digits` của sklearn; ta "giấu" nhãn để mô phỏng; dùng để học.

## Bản đồ các chặng

| Chặng | Câu hỏi của chặng |
|---|---|
| Chặng 0 | Thấy ngay: chỉ 30 nhãn thì được bao nhiêu |
| Chặng 1 | Tự dạy: giữ những nhãn đoán chắc chắn |
| Chặng 2 | Lan truyền nhãn qua hàng xóm |
| Chặng 3 | Bẫy: nhãn tự gán sai làm hỏng mô hình |
| Chặng 4 | Bao nhiêu nhãn thì đủ |
| Chặng 5 | Học chủ động: chọn ảnh gán nhãn tiếp theo |

## Chặng 0: Thấy ngay, chỉ 30 nhãn

**Câu hỏi:** mô hình học từ 30 ảnh có nhãn đạt bao nhiêu, so với khi có đủ nhãn?

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
BLUE, GRAY = "#1F4E79", "#8C8C8C"
plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

**Giải thích từng dòng**

- `import numpy as np`: mượn thư viện numpy (máy tính bỏ túi cho cả dãy số), đặt tên ngắn là `np` để gõ cho nhanh.
- `import pandas as pd`: mượn pandas, thư viện làm việc với bảng số liệu giống Excel, tên ngắn là `pd`.
- `import matplotlib.pyplot as plt`: mượn công cụ vẽ biểu đồ, tên ngắn là `plt`.
- `BLUE, GRAY = "#1F4E79", "#8C8C8C"`: gán hai màu dùng suốt bài (xanh đậm và xám). Một dòng gán cùng lúc hai biến, mỗi biến một mã màu.
- `plt.rcParams["figure.figsize"] = (6, 4)`: đặt cỡ hình mặc định: rộng 6, cao 4 (đơn vị inch).
- `plt.rcParams["axes.spines.top"] = False`: bỏ đường viền phía trên khung hình cho gọn mắt.
- `plt.rcParams["axes.spines.right"] = False`: bỏ luôn đường viền bên phải.

In [2]:
import warnings
from sklearn.semi_supervised import LabelSpreading
warnings.filterwarnings("ignore", message="invalid value encountered in divide", category=RuntimeWarning)
def spread(X, y_partial, k=10, alpha=0.2):
    return LabelSpreading(kernel="knn", n_neighbors=k, alpha=alpha, max_iter=200).fit(X, y_partial)
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
digits = load_digits()
X_pool, X_te, y_pool, y_te = train_test_split(digits.data, digits.target, test_size=0.25, random_state=0, stratify=digits.target)
sc = StandardScaler().fit(X_pool)
Xp, Xt = sc.transform(X_pool), sc.transform(X_te)
rng = np.random.RandomState(0)
labeled = np.concatenate([rng.choice(np.where(y_pool == d)[0], 3, replace=False) for d in range(10)])
unlabeled = np.setdiff1d(np.arange(len(Xp)), labeled)
print("pool:", len(Xp), "| labeled:", len(labeled), "| unlabeled:", len(unlabeled), "| test:", len(Xt))
small = LogisticRegression(max_iter=2000).fit(Xp[labeled], y_pool[labeled])
full = LogisticRegression(max_iter=2000).fit(Xp, y_pool)
acc_small, acc_full = small.score(Xt, y_te), full.score(Xt, y_te)
print("logistic on 30 labels:", round(acc_small, 3), "| on all", len(Xp), "labels (upper bound):", round(acc_full, 3))

pool: 1347 | labeled: 30 | unlabeled: 1317 | test: 450
logistic on 30 labels: 0.769 | on all 1347 labels (upper bound): 0.969


**Giải thích từng dòng**

- `warnings.filterwarnings(...)`: tắt đúng một cảnh báo "chia cho 0" của `LabelSpreading`: vài ảnh không có hàng xóm nối tới nên sklearn gặp 0 chia 0 cho riêng chúng.
- `def spread(...)`: hàm nhỏ tạo `LabelSpreading` để gọi cho gọn ở các chặng sau.
- `train_test_split(..., test_size=0.25)`: tách 25% làm tập thi có **nhãn thật** để chấm; phần còn lại là "kho" ảnh của chị Hạnh.
- `rng.choice(np.where(y_pool == d)[0], 3, replace=False)`: lấy ngẫu nhiên 3 ảnh mỗi chữ số làm 30 ảnh "đã gán nhãn". Cân bằng nhóm để công bằng.
- `np.setdiff1d(np.arange(len(Xp)), labeled)`: phần còn lại coi như **chưa gán nhãn**: ta sẽ không đưa nhãn của chúng cho mô hình.
- `full = LogisticRegression().fit(Xp, y_pool)`: mốc trần: nếu có đủ nhãn cho mọi ảnh thì đạt bao nhiêu.

**Cách đọc.**
- **Con số:** học từ 30 nhãn đạt **76,9%**; với đủ **1347** nhãn đạt **96,9%**. Khoảng cách **20,0** điểm phần trăm là phần ta hy vọng thu hẹp bằng **1317** ảnh chưa gán nhãn.
- **Câu hỏi tiếp:** mô hình có thể tự đoán nhãn cho các ảnh kia rồi học từ chính nó không?

**Mang về chặng 1:** ít nhãn thì mô hình yếu hơn mốc trần; mục tiêu là thu hẹp khoảng cách.

## Chặng 1: Tự dạy

**Câu hỏi:** mô hình đoán nhãn cho ảnh chưa gán, giữ phần chắc chắn, học lại; có tốt lên không?

In [3]:
from sklearn.semi_supervised import SelfTrainingClassifier
y_semi = y_pool.copy(); y_semi[unlabeled] = -1
rows = []
for thr in [0.6, 0.8, 0.95]:
    st = SelfTrainingClassifier(LogisticRegression(max_iter=2000), threshold=thr, max_iter=15).fit(Xp, y_semi)
    n_new = int((st.transduction_[unlabeled] != -1).sum())
    pseudo_acc = float((st.transduction_[unlabeled] == y_pool[unlabeled])[st.transduction_[unlabeled] != -1].mean())
    rows.append({"threshold": thr, "pseudo-labelled": n_new, "pseudo-label accuracy": round(pseudo_acc, 3), "test accuracy": round(st.score(Xt, y_te), 3), "rounds": int(st.n_iter_)})
st_table = pd.DataFrame(rows).set_index("threshold")
print(st_table.to_string())
print("baseline (30 labels only):", round(acc_small, 3))

           pseudo-labelled  pseudo-label accuracy  test accuracy  rounds
threshold                                                               
0.60                  1296                  0.860          0.864       6
0.80                  1221                  0.916          0.898      11
0.95                   759                  0.929          0.764      15
baseline (30 labels only): 0.769


**Giải thích từng dòng**

- `y_semi[unlabeled] = -1`: quy ước của sklearn: nhãn `-1` nghĩa là **chưa có nhãn**.
- `SelfTrainingClassifier(LogisticRegression(), threshold=thr)`: mỗi vòng: học từ ảnh đã có nhãn, đoán ảnh chưa nhãn, ảnh nào có xác suất đoán cao hơn `threshold` thì **nhận nhãn đoán làm nhãn thật**, lặp lại.
- `st.transduction_[unlabeled]`: nhãn cuối cùng của các ảnh chưa gán (vẫn là `-1` nếu chưa bao giờ đủ chắc chắn).
- `pseudo_acc`: trong những ảnh được tự gán nhãn, bao nhiêu gán **đúng** (ta biết vì đã giấu nhãn thật).

**Cách đọc.**
- **Con số:** ngưỡng 0,95: tự gán **759** ảnh với độ đúng **92,9%**, điểm thi **76,4%**; ngưỡng 0,6: tự gán **1296** ảnh với độ đúng **86,0%**, điểm thi **86,4%**. Mốc chỉ 30 nhãn: **76,9%**.
- **Bức tranh:** ngưỡng cao thì gán ít nhưng đúng nhiều; ngưỡng thấp thì gán nhiều nhưng lẫn nhãn sai. Điểm thi cho biết đánh đổi đó có đáng không: cao nhất ở ngưỡng **0.8** (**89,8%**). Ngưỡng 0,95 quá khắt (hết 15 vòng vẫn chỉ gán 759 ảnh) nên điểm thi không hơn mốc 30 nhãn; ngưỡng quá thấp thì nhãn giả lẫn nhiều lỗi. Tự dạy **không phải lúc nào cũng thắng**: cần chọn ngưỡng.
- **Câu hỏi tiếp:** có cách dùng cấu trúc dữ liệu, ảnh nào gần ảnh nào, thay vì chỉ dựa vào mô hình?

**Mang về chặng 2:** tự dạy là đoán chắc rồi học lại; ngưỡng cân đối số lượng và độ đúng của nhãn mới.

## Chặng 2: Lan truyền nhãn

**Câu hỏi:** nhãn "chảy" qua mạng hàng xóm giữa các ảnh có giúp hơn không?

In [4]:
rows = []
for k in [5, 10, 20]:
    ls = spread(Xp, y_semi, k=k)
    rows.append({"neighbors": k, "accuracy on unlabeled pool": round(float((ls.transduction_[unlabeled] == y_pool[unlabeled]).mean()), 3), "test accuracy": round(ls.score(Xt, y_te), 3)})
ls_table = pd.DataFrame(rows).set_index("neighbors")
print(ls_table.to_string())
print("baseline (30 labels only):", round(acc_small, 3), "| upper bound:", round(acc_full, 3))

           accuracy on unlabeled pool  test accuracy
neighbors                                           
5                               0.698          0.791
10                              0.818          0.867
20                              0.826          0.869
baseline (30 labels only): 0.769 | upper bound: 0.969


**Giải thích từng dòng**

- `LabelSpreading(kernel="knn", n_neighbors=k, alpha=0.2)`: nối mỗi ảnh với `k` ảnh giống nó nhất (như k-NN, C3) thành một mạng, rồi cho nhãn "lan" dọc các cạnh. `alpha` quyết định mức nhãn gốc bị pha loãng khi lan: nhỏ thì giữ chặt nhãn gốc.
- `ls.transduction_`: nhãn mà thuật toán gán cho **mọi ảnh trong kho**, kể cả ảnh chưa gán.
- `ls.score(Xt, y_te)`: điểm trên tập thi riêng (ảnh chưa từng thấy), bằng cách nhìn xem ảnh thi gần ảnh nào.

**Cách đọc.**
- **Con số:** điểm thi theo số hàng xóm: k=5 **79,1%**, k=10 **86,7%**, k=20 **86,9%**; mốc 30 nhãn **76,9%**, mốc trần **96,9%**.
- **Bức tranh:** lan truyền nhãn dựa vào giả định "ảnh giống nhau thì cùng nhãn" và dùng tất cả 1347 ảnh để dựng mạng hàng xóm. Với chữ số, các ảnh cùng số tụ lại thành đám (đã thấy ở F5), nên giả định này đúng và nhãn lan đúng đám.
- **Quy tắc đọc:** hợp khi dữ liệu có cấu trúc đám rõ; kém khi các nhóm chồng lấn hoặc khi các cột nhiễu làm khoảng cách mất ý nghĩa.

**Mang về chặng 3:** lan truyền nhãn dùng cấu trúc hàng xóm; chính xác khi giả định "giống nhau thì cùng nhãn" đúng.

## Chặng 3: Bẫy, nhãn tự gán sai làm hỏng mô hình

**Câu hỏi:** nếu mô hình đoán sai rồi học theo cái sai, chuyện gì xảy ra?

In [5]:
bad_start = np.concatenate([rng.choice(np.where(y_pool == d)[0], 1, replace=False) for d in range(10)])
y_bad = y_pool.copy(); y_bad[np.setdiff1d(np.arange(len(Xp)), bad_start)] = -1
wrong = rng.choice(len(bad_start), 3, replace=False)
y_bad_noisy = y_bad.copy()
for i in wrong:
    y_bad_noisy[bad_start[i]] = (y_pool[bad_start[i]] + 1) % 10
rows = []
for name, yy in [("10 labels, all correct", y_bad), ("10 labels, 3 of them wrong", y_bad_noisy)]:
    base = LogisticRegression(max_iter=2000).fit(Xp[bad_start], yy[bad_start])
    st = SelfTrainingClassifier(LogisticRegression(max_iter=2000), threshold=0.8, max_iter=15).fit(Xp, yy)
    ls = spread(Xp, yy, k=10)
    rows.append({"setup": name, "supervised only": round(base.score(Xt, y_te), 3), "self-training": round(st.score(Xt, y_te), 3), "label spreading": round(ls.score(Xt, y_te), 3)})
bad_table = pd.DataFrame(rows).set_index("setup")
print(bad_table.to_string())

                            supervised only  self-training  label spreading
setup                                                                      
10 labels, all correct                0.511          0.544            0.736
10 labels, 3 of them wrong            0.338          0.440            0.562


**Giải thích từng dòng**

- `rng.choice(np.where(y_pool == d)[0], 1, ...)`: lần này chỉ 1 nhãn mỗi chữ số (10 nhãn), để thấy rõ hậu quả của nhãn sai.
- `y_bad_noisy[...] = (y_pool[...] + 1) % 10`: cố ý **gán sai** 3 trong 10 nhãn (nhầm sang chữ số kế bên), như người gán nhãn bấm nhầm.

**Cách đọc.**
- **Con số:** khi 10 nhãn đều đúng: chỉ học có nhãn **51,1%**, tự dạy **54,4%**, lan truyền **73,6%**. Khi 3 trong 10 nhãn sai: **33,8%**, **44,0%**, **56,2%**.
- **Bức tranh:** với rất ít nhãn, mỗi nhãn sai có sức nặng lớn; các cách bán giám sát không tự sửa được nhãn gốc sai mà còn **lan chúng ra** cho hàng trăm ảnh hàng xóm. Dữ liệu chưa nhãn khuếch đại cả cái đúng lẫn cái sai (dù các cách bán giám sát vẫn hơn mô hình chỉ có nhãn ở cả hai hàng).
- **Quy tắc đọc:** với ít nhãn, **kiểm lại từng nhãn gốc** quan trọng hơn thêm thuật toán. Luôn đo trên tập thi có nhãn do người kiểm.

**Mang về chặng 4:** dữ liệu chưa nhãn khuếch đại cả nhãn đúng lẫn nhãn sai.

## Chặng 4: Bao nhiêu nhãn thì đủ

**Câu hỏi:** khi có thêm nhãn, lợi thế của bán giám sát còn không?

In [6]:
rows = []
for per_class in [1, 2, 3, 5, 10, 30]:
    idx = np.concatenate([np.random.RandomState(7).choice(np.where(y_pool == d)[0], per_class, replace=False) for d in range(10)])
    yy = np.full_like(y_pool, -1); yy[idx] = y_pool[idx]
    sup = LogisticRegression(max_iter=2000).fit(Xp[idx], y_pool[idx])
    ls = spread(Xp, yy, k=10)
    rows.append({"labels": len(idx), "supervised": round(sup.score(Xt, y_te), 3), "label spreading": round(ls.score(Xt, y_te), 3)})
lab_curve = pd.DataFrame(rows).set_index("labels")
lab_curve["gain (points)"] = ((lab_curve["label spreading"] - lab_curve["supervised"]) * 100).round(1)
print(lab_curve.to_string())

        supervised  label spreading  gain (points)
labels                                            
10           0.644            0.731            8.7
20           0.727            0.782            5.5
30           0.796            0.862            6.6
50           0.818            0.902            8.4
100          0.869            0.927            5.8
300          0.933            0.947            1.4


**Giải thích từng dòng**

- `for per_class in [1, 2, 3, 5, 10, 30]`: từ 10 đến 300 nhãn (1 đến 30 nhãn mỗi chữ số).
- `RandomState(7)`: cố định hạt giống để mỗi cỡ nhãn lấy cùng cách; đây vẫn chỉ là **một** lần chọn ngẫu nhiên, nên số liệu có dao động.
- `gain (points)`: điểm cộng của lan truyền nhãn so với chỉ học có nhãn.

**Cách đọc.**
- **Con số:** với **10** nhãn, lan truyền nhãn hơn học có nhãn **8,7** điểm; với **50** nhãn **8,4** điểm; với **300** nhãn **1,4** điểm.
- **Bức tranh:** trong khoảng 10 đến 100 nhãn lan truyền nhãn luôn hơn học có nhãn, cỡ **5,5** đến **8,7** điểm (không giảm đều vì mỗi dòng là một lần chọn nhãn ngẫu nhiên). Khi nhãn lên **300**, lợi thế co lại còn **1,4** điểm: có nhiều nhãn rồi thì mô hình có nhãn tự học đủ.
- **Lưu ý:** mỗi dòng là một lần chọn nhãn ngẫu nhiên; chênh lệch vài điểm giữa hai dòng liền kề có thể là may rủi.
- **Quy tắc đọc:** bán giám sát đáng thử khi **nhãn đắt, dữ liệu thô rẻ**; khi nhãn đã đủ nhiều, lợi ích nhỏ dần.

**Mang về chặng 5:** bán giám sát giúp rõ khi nhãn khan hiếm, và nhỏ dần khi nhãn dồi dào.

## Chặng 5: Học chủ động, chọn ảnh gán nhãn tiếp theo

**Câu hỏi:** nếu chị Hạnh gán thêm 5 ảnh mỗi vòng, chọn ảnh nào thì mô hình tốt lên nhanh nhất?

In [7]:
def run_active(strategy, seed=0, rounds=8, per_round=5):
    r = np.random.RandomState(seed)
    idx = list(np.concatenate([r.choice(np.where(y_pool == d)[0], 1, replace=False) for d in range(10)]))
    curve = []
    for _ in range(rounds):
        m = LogisticRegression(max_iter=2000).fit(Xp[idx], y_pool[idx])
        curve.append(m.score(Xt, y_te))
        rest = np.setdiff1d(np.arange(len(Xp)), idx)
        if strategy == "random":
            pick = r.choice(rest, per_round, replace=False)
        else:
            proba = m.predict_proba(Xp[rest])
            top2 = np.sort(proba, axis=1)[:, -2:]
            margin = top2[:, 1] - top2[:, 0]
            pick = rest[np.argsort(margin)[:per_round]]
        idx += list(pick)
    return np.array(curve)
seeds = range(5)
rand_c = np.mean([run_active("random", s) for s in seeds], axis=0)
unc_c = np.mean([run_active("uncertain", s) for s in seeds], axis=0)
act = pd.DataFrame({"labels": [10 + 5 * i for i in range(8)], "random": rand_c.round(3), "least confident": unc_c.round(3)}).set_index("labels")
print(act.to_string())

        random  least confident
labels                         
10       0.576            0.576
15       0.608            0.597
20       0.673            0.649
25       0.715            0.723
30       0.735            0.736
35       0.754            0.786
40       0.782            0.810
45       0.803            0.852


**Giải thích từng dòng**

- `strategy == "random"`: cách 1: chọn ngẫu nhiên ảnh để gán nhãn.
- `np.sort(proba, axis=1)[:, -2:]`: lấy hai xác suất cao nhất của mỗi ảnh.
- `margin = top2[:, 1] - top2[:, 0]`: **độ chênh** giữa lựa chọn thứ nhất và thứ nhì. Chênh nhỏ nghĩa là mô hình đang **phân vân**.
- `rest[np.argsort(margin)[:per_round]]`: cách 2: chọn những ảnh mô hình phân vân nhất để người gán nhãn. Ảnh đó mang nhiều thông tin nhất.
- `for s in seeds ... np.mean(...)`: lặp 5 hạt giống và lấy trung bình, vì một lần chạy dao động nhiều (G2, chặng 4).

**Cách đọc.**
- **Con số:** sau **45** nhãn, chọn ngẫu nhiên đạt **80,3%**, chọn ảnh phân vân nhất đạt **85,2%**. Ở giữa chặng (**25** nhãn): **71,5%** so với **72,3%**.
- **Bức tranh:** chọn ảnh phân vân giúp mô hình tốt lên nhanh hơn so với gán nhãn ngẫu nhiên, cùng số công gán nhãn. Ở vài vòng đầu (từ 15 đến 20 nhãn) hai cách ngang nhau hoặc cách phân vân hơi kém, lợi thế chỉ lộ ra khi mô hình đã khá hơn (từ khoảng 35 nhãn). Ý tưởng: không cần gán thêm ảnh mà mô hình đã chắc, hãy gán chỗ nó còn phân vân.
- **Lưu ý:** học chủ động có thể lệch: nó chỉ hỏi về chỗ khó nên tập nhãn không còn đại diện cho phân bố thật. Vẫn cần tập thi ngẫu nhiên để chấm.

**Mang về:** có ít nhãn: kiểm nhãn gốc, thử lan truyền hoặc tự dạy, và chọn ảnh gán thêm theo độ phân vân.

## Tổng kết

**Mang về theo từng chặng**

| Chặng | Mang về |
|---|---|
| 0 | Ít nhãn thì mô hình yếu hơn mốc trần; dữ liệu chưa nhãn có thể thu hẹp khoảng cách. |
| 1 | Tự dạy: giữ nhãn đoán chắc chắn; ngưỡng cân đối số lượng và độ đúng. |
| 2 | Lan truyền nhãn dùng cấu trúc hàng xóm; hợp khi dữ liệu có đám rõ. |
| 3 | Dữ liệu chưa nhãn khuếch đại cả nhãn đúng lẫn nhãn sai; kiểm nhãn gốc. |
| 4 | Lợi ích bán giám sát lớn nhất khi nhãn khan hiếm. |
| 5 | Học chủ động: gán nhãn chỗ mô hình phân vân; vẫn cần tập thi ngẫu nhiên. |

**Bảng đối chiếu: đời thường, thuật ngữ, tên trong code**

| Đời thường | Thuật ngữ | Tên trong code |
|---|---|---|
| Chưa có sổ điểm | chưa gán nhãn | nhãn `-1` |
| Đoán chắc rồi coi như đã biết | nhãn giả | `SelfTrainingClassifier` |
| Hàng xóm giống nhau cùng nhãn | lan truyền nhãn | `LabelSpreading` |
| Ngưỡng đủ chắc để tin | ngưỡng tự tin | `threshold` |
| Hỏi người chỗ mình còn phân vân | học chủ động | `margin`, `argsort` |
| Tập thi do người kiểm | tập thi có nhãn thật | `X_te`, `y_te` |

**Thử thách tự làm**

1. Ở chặng 1, thử `threshold=0.99`. Số nhãn giả và điểm thi thay đổi thế nào?
2. Ở chặng 2, thử `kernel='rbf'` thay cho `knn` (với `gamma=0.01`). Có hơn không?
3. Ở chặng 3, gán sai 1 nhãn thay vì 3. Ảnh hưởng nhỏ hơn bao nhiêu?
4. Ở chặng 5, thử chọn theo `entropy` thay cho độ chênh. Hai cách có cho kết quả khác không?

**Checklist năng lực** (tự tick khi làm được)

- [ ] Giải thích vì sao ảnh chưa nhãn có thể giúp.
- [ ] Dùng tự dạy và lan truyền nhãn với nhãn `-1`.
- [ ] Nhận ra nhãn sai bị khuếch đại.
- [ ] So bán giám sát với mô hình chỉ có nhãn làm mốc.
- [ ] Chọn ảnh gán nhãn tiếp theo theo độ phân vân.

## Phần thưởng (Tip pro): Sang G7: thực hành tổng hợp phần nâng cao


G7 gom lại những gì Phần G đã dạy vào một bài toán trọn vẹn: mạng nơ-ron hoặc mô hình khác, cột mới, chọn cột, khoảng dự đoán, trong một pipeline kiểm định chặt chẽ.